# Week 4 — Capstone end-to-end demo

Runs the full LangGraph pipeline against the FakeLLMClient for a single ticker, then exercises the backtest helpers on the synthetic prices.

In [ ]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from invest_forge.agents.graph import GraphDeps, run_pipeline_inline
from invest_forge.tools.data_tools import FakeDataProvider
from invest_forge.tools.sentiment import LexiconSentiment
from invest_forge.knowledge_base.builder import build_in_memory
from invest_forge.llm.fake import FakeLLMClient
deps = GraphDeps(
    llm=FakeLLMClient(),
    data_provider=FakeDataProvider(sample_dir=ROOT/'data'/'sample'),
    sentiment=LexiconSentiment(),
    retriever=build_in_memory(ROOT/'data'/'sample'/'kb'),
)
state = run_pipeline_inline(deps, ts_code='688981.SH')
print(json.dumps(state['final_recommendation'], ensure_ascii=False, indent=2))


In [ ]:
import pandas as pd
from invest_forge.tools.backtest_tools import run_simple_backtest
prices = pd.read_csv(ROOT/'data'/'sample'/'prices.csv')
for code, grp in prices.groupby('ts_code'):
    result = run_simple_backtest(grp.sort_values('trade_date')['close'])
    print(f'{code}  sharpe={result.sharpe:+.2f}  ic={result.ic_mean:+.3f}  mdd={result.max_drawdown:.2%}')
